# USDA Phytochemical Database — Quickstart

This notebook demonstrates 5 practical use cases with the **free 400-row sample**.

**Full dataset (104,388 rows):** [ethno-api.com](https://ethno-api.com)  
**GitHub:** [wirthal1990-tech/USDA-Phytochemical-Database-JSON](https://github.com/wirthal1990-tech/USDA-Phytochemical-Database-JSON)

---

In [ ]:
# Cell 2 — Load the 400-row sample
import pandas as pd

url = "https://raw.githubusercontent.com/wirthal1990-tech/USDA-Phytochemical-Database-JSON/main/ethno_sample_400.json"
df = pd.read_json(url)
print(f"Records: {df.shape[0]}")
print(f"Unique compounds: {df['chemical'].nunique()}")
print(f"Unique species: {df['plant_species'].nunique()}")
print(f"\nSchema:")
print(df.dtypes)
df.head()

In [ ]:
# Cell 3 — Top 10 compounds by PubMed mentions (with bar chart)
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 5)

top10 = (
    df.groupby('chemical')['pubmed_mentions_2026']
    .first()
    .nlargest(10)
    .sort_values(ascending=True)
)

top10.plot.barh(color='#2563eb')
import matplotlib.pyplot as plt
plt.xlabel('PubMed Mentions (March 2026)')
plt.title('Top 10 Most-Studied Phytochemicals')
plt.tight_layout()
plt.show()

print(top10.to_frame().to_string())

In [ ]:
# Cell 4 — DuckDB: Anti-inflammatory compounds ranked by clinical evidence
# pip install duckdb  (once)
import duckdb

# Query directly from the pandas DataFrame loaded in Cell 2
result = duckdb.sql("""
    SELECT
        chemical,
        MAX(pubmed_mentions_2026)  AS pubmed_score,
        COUNT(DISTINCT plant_species) AS species_count
    FROM df
    WHERE application ILIKE '%anti%inflam%'
       OR application ILIKE '%antiinflam%'
    GROUP BY chemical
    ORDER BY pubmed_score DESC
    LIMIT 15
""")
print("Top anti-inflammatory compounds by PubMed evidence:")
result.show()

In [ ]:
# Cell 5 — RAG pipeline use case: build a deterministic compound context block
# This is the pattern used to ground LLMs with plant chemistry data.

query_compound = "QUERCETIN"

# Filter all records for this compound
compound_records = df[df["chemical"] == query_compound].copy()
compound_records = compound_records.sort_values("pubmed_mentions_2026", ascending=False)

print(f"=== RAG CONTEXT BLOCK FOR: {query_compound} ===\n")
for _, row in compound_records.head(5).iterrows():
    block = (
        f"Compound: {row['chemical']}\n"
        f"Plant: {row['plant_species']}\n"
        f"Application: {row['application']}\n"
        f"Dosage: {row['dosage']}\n"
        f"PubMed mentions (2026): {row['pubmed_mentions_2026']}\n"
        f"---"
    )
    print(block)

print(f"\nTotal records for {query_compound}: {len(compound_records)}")
print("These structured blocks eliminate LLM hallucinations about botanical dosages.")

In [ ]:
# Cell 6 — Parquet vs JSON: memory and size comparison
import io

json_bytes   = df.to_json(orient="records").encode("utf-8")
parquet_buf  = io.BytesIO()
df.to_parquet(parquet_buf, engine="pyarrow", compression="snappy")
parquet_size = parquet_buf.tell()
json_size    = len(json_bytes)

print(f"{'Format':<10} {'Size':>12}  {'Ratio':>8}")
print(f"{'JSON':<10} {json_size:>12,} B  {'100.0%':>8}")
print(f"{'Parquet':<10} {parquet_size:>12,} B  {parquet_size/json_size:>8.1%}")
print()
print(f"Parquet is {json_size/parquet_size:.1f}× smaller than JSON for this sample.")
print()
print("Full 104,388-row dataset:")
print("  JSON:    ~16.4 MB  (ethno_dataset_v2.json)")
print("  Parquet: ~761 KB   (ethno_dataset_v2.parquet, Snappy-compressed)")
print("  Ratio:   ~22× smaller")

---

## Get the Full Dataset

| | Free Sample | Full Dataset |
|---|---|---|
| **Records** | 400 | 104,388 |
| **Schema** | 8 columns | 8 columns |
| **Enrichment fields** | Placeholder values | Real values (CT, ChEMBL, Patents) |
| **Price** | Free | €699 one-time |

**Purchase:** [ethno-api.com](https://ethno-api.com)  
**Questions:** founder@ethno-api.com